<a href="https://colab.research.google.com/github/R786P/telco_churn_predictor/blob/main/fooocus_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pygit2==1.15.1
%cd /content
!git clone https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus
!python entry_with_update.py --share --always-high-vram


In [1]:
import pandas as pd

def clean_churn_data(df: pd.DataFrame) -> pd.DataFrame:
    """Cleans raw Telco churn dataset"""
    df = df.copy()

    # 1. TotalCharges has spaces → convert to numeric
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # 2. Drop rows with missing critical values
    df = df.dropna(subset=["TotalCharges", "MonthlyCharges"])

    # 3. Encode target variable
    df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

    # 4. Remove non-predictive ID column
    df = df.drop(columns=["customerID"])

    return df.reset_index(drop=True)

In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
# from data_cleaning import clean_churn_data # This line is removed
# from feature_engineering import build_preprocessor # This line is removed

def main():
    # 1. Load & Clean
    df = clean_churn_data(pd.read_csv("data/raw/churn.csv"))
    X = df.drop("Churn", axis=1)
    y = df["Churn"]

    # 2. Stratified Split (important for imbalanced data)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Preprocess
    preprocessor, _, _ = build_preprocessor(X_train)
    X_train_prep = preprocessor.fit_transform(X_train)
    X_test_prep = preprocessor.transform(X_test)

    # 4. Train Model
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",  # handles churn imbalance
        n_jobs=-1
    )
    model.fit(X_train_prep, y_train)

    # 5. Save Artifact
    artifact = {
        "model": model,
        "preprocessor": preprocessor,
        "feature_names": list(X.columns)
    }
    joblib.dump(artifact, "models/churn_rf_v1.pkl")
    print("✅ Model trained & saved to models/churn_rf_v1.pkl")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'feature_engineering'